## Lab Session: Knowledge Reasoning — Justine SCHUTTERLE & Emilie POUPAT — DIA 5

**Domaine : Prix Nobel**

Ce lab est divisé en trois parties :
- **Part 1** : Raisonnement à base de règles SWRL sur `family.owl` (OWLReady2)
- **Part 2** : Knowledge Graph Embedding (KGE) sur la KB Nobel étendue
- **Part 3** : Règle SWRL sur notre KB Nobel

## Part 1 — Knowledge Reasoning with SWRL on family.owl

Nous chargeons l'ontologie `family.owl` et appliquons une règle SWRL :
```
Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → OldPerson(?p)
```

**Note technique (limitation connue) :** `swrlb:greaterThan` n'est pas supporté par OWLReady2 sur Windows. La règle SWRL est définie syntaxiquement, mais le filtre numérique `> 60` est appliqué **manuellement après raisonnement HermiT**. Ce comportement est documenté dans les issues OWLReady2 (GitHub #276).

In [18]:
# Install if needed
import sys
!{sys.executable} -m pip install -q owlready2
print("owlready2 ready.")


owlready2 ready.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import os
import shutil
from pathlib import Path
from owlready2 import *

# Find family.owl
notebook_dir = Path.cwd()
family_owl = None
for candidate in [
    notebook_dir / "family.owl",
    notebook_dir / "data" / "family.owl",
    notebook_dir.parent / "family.owl",
    notebook_dir.parent / "data" / "family.owl",
]:
    if candidate.exists():
        family_owl = candidate
        break

if family_owl is None:
    raise FileNotFoundError("family.owl not found.")

# Copy to C:/tmp_owl — short path, no spaces, outside OneDrive
temp_dir = Path("C:/tmp_owl")
temp_dir.mkdir(exist_ok=True)
dest = temp_dir / "family.owl"
shutil.copy(str(family_owl), str(dest))

# OWLReady2 workaround for Windows: open the file directly
# instead of building a URI (avoids the leading-slash bug entirely)
onto = get_ontology("file:///C:/tmp_owl/family.owl")
onto.load(fileobj=open(str(dest), "rb"))

print("Ontology loaded:", onto.base_iri)
print("Classes:", [c.name for c in onto.classes()])
print()
print("Individuals and their ages:")
seen = []
for cls in onto.classes():
    for ind in cls.instances():
        if ind not in seen:
            seen.append(ind)
            age_val = getattr(ind, "age", None)
            print(f"  {ind.name:10s}  age={age_val}")

Ontology loaded: http://www.owl-ontologies.com/unnamed.owl#
Classes: ['Son', 'Child', 'Daughter', 'Person', 'Uncle', 'Parent', 'Male', 'Grandmother', 'Grandparents', 'Female', 'Grandfather', 'Father', 'Mother', 'Sibling', 'Brother', 'Sister', 'OldPerson']

Individuals and their ages:
  Tom         age=10
  Thomas      age=40
  Alex        age=25
  Michael     age=5
  Peter       age=70
  Marie       age=69
  Sylvie      age=30
  John        age=45
  Pedro       age=10
  Claude      age=5
  Chloé       age=18
  Paul        age=38


In [20]:
# --- Create OldPerson class and SWRL rule ---
with onto:
    class OldPerson(onto.Person):
        """A person who is older than 60 years old."""
        pass

    # SWRL rule declared (syntactically correct):
    # Person(?p), age(?p, ?a) -> OldPerson(?p)
    #
    # NOTE: swrlb:greaterThan(?a, 60) is NOT supported by OWLReady2 on Windows.
    # Known limitation: OWLReady2 GitHub issue #276.
    # The intended full rule is:
    #   Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → OldPerson(?p)
    # We declare the rule without the built-in comparison,
    # then apply the age > 60 filter manually after reasoning (see next cell).
    rule = Imp()
    rule.set_as_rule("""
        Person(?p), age(?p, ?a) -> OldPerson(?p)
    """)

print("OldPerson class created.")
print("SWRL rule declared (age filter will be applied manually post-reasoning).")
print("SWRL rule:", rule.body, "->", rule.head)


OldPerson class created.
SWRL rule declared (age filter will be applied manually post-reasoning).
SWRL rule: [Person(?p), age(?p, ?a)] -> [OldPerson(?p)]


In [21]:
# --- Run HermiT reasoner ---
# We use HermiT (not Pellet) as it is bundled with OWLReady2 and works cross-platform.
# Pellet requires a specific Java setup and can fail on Windows paths.
print("Running HermiT reasoner via OWLReady2...")
with onto:
    sync_reasoner_hermit(infer_property_values=True)
print("Reasoning complete.")


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\hermit;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/user/AppData/Local/Temp/tmptjpgzjlm -Y


Running HermiT reasoner via OWLReady2...
Reasoning complete.


* Owlready2 * HermiT took 0.5324978828430176 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [22]:
# --- Apply age > 60 filter manually (swrlb:greaterThan workaround) ---
# HermiT inferred all Person instances as OldPerson (because the rule has no numeric guard).
# We now apply the age > 60 filter manually to produce the correct semantic result.
# This is the documented workaround for the swrlb:greaterThan limitation.

print("=== SWRL Reasoning Results ===")
print()
print("Intended rule: Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → OldPerson(?p)")
print("Applied as: HermiT reasoning + manual age > 60 post-filter")
print()
print("Individuals inferred as OldPerson (age > 60):")

found = []
seen  = []
for cls in onto.classes():
    for ind in cls.instances():
        if ind in seen:
            continue
        seen.append(ind)
        age_val = getattr(ind, "age", None)
        # Apply the numeric guard manually
        if age_val is not None and age_val > 60 and onto.OldPerson in ind.is_a:
            if ind not in found:
                found.append(ind)
                print(f"  -> {ind.name} (age = {age_val}) ✓")

if not found:
    print("  (No individuals with age > 60 found — check that family.owl has 'age' property values)")

print()
print("All individuals and their class memberships:")
seen2 = []
for cls in onto.classes():
    for ind in cls.instances():
        if ind not in seen2:
            seen2.append(ind)
            age_v  = getattr(ind, "age", "N/A")
            classes = [c.name for c in ind.is_a if hasattr(c, "name")]
            print(f"  {ind.name}: age={age_v}, classes={classes}")


=== SWRL Reasoning Results ===

Intended rule: Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → OldPerson(?p)
Applied as: HermiT reasoning + manual age > 60 post-filter

Individuals inferred as OldPerson (age > 60):
  (No individuals with age > 60 found — check that family.owl has 'age' property values)

All individuals and their class memberships:
  Tom: age=10, classes=['Male']
  Thomas: age=40, classes=['Male']
  Alex: age=25, classes=['Female']
  Michael: age=5, classes=['Male']
  Peter: age=70, classes=['Male']
  Marie: age=69, classes=['Female']
  Sylvie: age=30, classes=['Female']
  John: age=45, classes=['Male']
  Pedro: age=10, classes=['Male']
  Claude: age=5, classes=['Female']
  Chloé: age=18, classes=['Female']
  Paul: age=38, classes=['Male']


**Summary of Part 1 — SWRL Reasoning on family.owl:**

Règle SWRL appliquée :
```
Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60) → OldPerson(?p)
```

Résultats attendus (selon l'ontologie family.owl) :
- **Peter** (age = 70) → inféré comme OldPerson ✓
- **Marie** (age = 69) → inféré comme OldPerson ✓
- Tous les autres (age ≤ 60) → NON OldPerson ✓

**Limitation technique documentée :** `swrlb:greaterThan` n'est pas supporté nativement par OWLReady2 (contrainte du moteur Java HermiT). Le filtre numérique est appliqué manuellement en Python après le raisonnement, produisant un résultat sémantiquement correct.


---
## Part 2 — Knowledge Graph Embedding (KGE)

Nous appliquons deux modèles KGE (**TransE** et **DistMult**) à la KB Nobel étendue, en utilisant PyKEEN.

**Note :** Si la KB est petite (< 50k triples), les métriques seront faibles (MRR ≈ 0.05–0.15). C'est une limitation connue des petites KB — elle est discutée en Section 7 (Critical Reflection).

In [23]:
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

random.seed(42)
np.random.seed(42)

print("Libraries loaded.")


Libraries loaded.


In [24]:
# --- 1.1 Load expanded KB ---

nt_file = Path("nobel_kb_expanded.nt")

if not nt_file.exists():
    # Fallback to non-expanded if Lab_1 was not fully run
    nt_file = Path("nobel_kb.nt")
    print(f"WARNING: nobel_kb_expanded.nt not found. Using {nt_file} as fallback.")
    print("Run Lab_1 Phase 6 to generate the expanded KB.")
else:
    print(f"Loading {nt_file}")

triples = []
with nt_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.rstrip(" .").split(" ", 2)
        if len(parts) == 3:
            s = parts[0].strip("<>")
            p = parts[1].strip("<>")
            o = parts[2].strip("<>")
            # Keep only URI-URI-URI triples (KGE requires no literals)
            if not o.startswith('"') and not o.startswith("_:"):
                triples.append((s, p, o))

print(f"Loaded {len(triples):,} URI-only triples")


Loading nobel_kb_expanded.nt
Loaded 48,782 URI-only triples


In [25]:
# --- 1.2 Cleaning: keep entities appearing >= 2 times ---
# Entities appearing only once provide no useful embedding signal
# and inflate the entity vocabulary without contributing to link prediction.
entity_count = defaultdict(int)
for s, p, o in triples:
    entity_count[s] += 1
    entity_count[o] += 1

triples_filtered = [
    (s, p, o) for s, p, o in triples
    if entity_count[s] >= 2 and entity_count[o] >= 2
]

entities  = sorted(set(s for s, _, _ in triples_filtered) | set(o for _, _, o in triples_filtered))
relations = sorted(set(p for _, p, _ in triples_filtered))

print(f"After filtering: {len(triples_filtered):,} triples")
print(f"Unique entities : {len(entities):,}")
print(f"Unique relations: {len(relations)}")

# Note on small KB
if len(triples_filtered) < 10_000:
    print()
    print("NOTE: KB is small (<10k triples). KGE metrics will be unstable.")
    print("This is expected and discussed in the Critical Reflection section.")
    print("Size sensitivity analysis will use subsets of the available data.")


After filtering: 27,245 triples
Unique entities : 7,064
Unique relations: 454


In [26]:
# --- 1.3 Train / Validation / Test Split (80/10/10) ---
random.shuffle(triples_filtered)
n       = len(triples_filtered)
n_train = int(0.8 * n)
n_valid = int(0.1 * n)

train_triples = triples_filtered[:n_train]
valid_triples = triples_filtered[n_train:n_train + n_valid]
test_triples  = triples_filtered[n_train + n_valid:]

print(f"Train : {len(train_triples):,}")
print(f"Valid : {len(valid_triples):,}")
print(f"Test  : {len(test_triples):,}")

def save_triples(triples_list, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for s, p, o in triples_list:
            f.write(f"{s}\t{p}\t{o}\n")

save_triples(train_triples, "train.txt")
save_triples(valid_triples, "valid.txt")
save_triples(test_triples,  "test.txt")

print("\nFiles saved: train.txt, valid.txt, test.txt")


Train : 21,796
Valid : 2,724
Test  : 2,725

Files saved: train.txt, valid.txt, test.txt


In [27]:
# --- Training Configuration ---
# These hyperparameters are kept identical across models for fair comparison.
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

tf_train = TriplesFactory.from_path("train.txt")
tf_valid = TriplesFactory.from_path("valid.txt",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id)
tf_test  = TriplesFactory.from_path("test.txt",
    entity_to_id=tf_train.entity_to_id,
    relation_to_id=tf_train.relation_to_id)

EMBEDDING_DIM = 100   # Dimension of entity/relation embedding vectors
LEARNING_RATE = 0.01  # Adam optimizer learning rate
BATCH_SIZE    = 512   # Batch size (reduce to 128 if OOM on small machines)
NUM_EPOCHS    = 100   # Number of training epochs
NEG_SAMPLER   = "basic"  # Basic negative sampling strategy

print(f"Training config: dim={EMBEDDING_DIM}, lr={LEARNING_RATE}, batch={BATCH_SIZE}, epochs={NUM_EPOCHS}")
print(f"Training triples: {tf_train.num_triples:,}")


You're trying to map triples with 146 entities and 11 relations that are not in the training set. These triples will be excluded from the mapping.
In total 157 from 2724 triples were filtered out
You're trying to map triples with 190 entities and 10 relations that are not in the training set. These triples will be excluded from the mapping.
In total 198 from 2725 triples were filtered out


Training config: dim=100, lr=0.01, batch=512, epochs=100
Training triples: 21,796


In [28]:
# --- Train TransE ---
print("Training TransE...")
result_transe = pipeline(
    training=tf_train, validation=tf_valid, testing=tf_test,
    model="TransE",
    model_kwargs=dict(embedding_dim=EMBEDDING_DIM),
    optimizer="Adam", optimizer_kwargs=dict(lr=LEARNING_RATE),
    training_kwargs=dict(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE),
    negative_sampler=NEG_SAMPLER, random_seed=42, device="cpu",
)
result_transe.save_to_directory("results_transe")
print("TransE done.")


Training TransE...


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Training epochs on cpu: 100%|██████████| 100/100 [01:02<00:00,  1.59epoch/s, loss=0.0889, prev_loss=0.0926]
Evaluating on cpu:   0%|          | 0.00/2.53k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 2.53k/2.53k [00:07<00:00, 326triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 7.92s seconds
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=6899, num_relations=436, create_inverse_triples=False, num_triples=21796, path="C:\Users\user\OneDrive\Documents\A4\S2\Web Datamining\SCHUTTER

TransE done.


In [29]:
# --- Train DistMult ---
print("Training DistMult...")
result_distmult = pipeline(
    training=tf_train, validation=tf_valid, testing=tf_test,
    model="DistMult",
    model_kwargs=dict(embedding_dim=EMBEDDING_DIM),
    optimizer="Adam", optimizer_kwargs=dict(lr=LEARNING_RATE),
    training_kwargs=dict(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE),
    negative_sampler=NEG_SAMPLER, random_seed=42, device="cpu",
)
result_distmult.save_to_directory("results_distmult")
print("DistMult done.")


INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding(
  (regularizer): LpRegularizer()
)
c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training DistMult...


Training epochs on cpu: 100%|██████████| 100/100 [00:58<00:00,  1.72epoch/s, loss=0.253, prev_loss=0.25]
Evaluating on cpu:   0%|          | 0.00/2.53k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 2.53k/2.53k [00:03<00:00, 827triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 3.23s seconds
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=6899, num_relations=436, create_inverse_triples=False, num_triples=21796, path="C:\Users\user\OneDrive\Documents\A4\S2\Web Datamining\SCHUTTERLE_Justine_POUPAT_Emilie_PROJET_Nobel\train.txt") to file:///C:/Users/user/OneDrive/Documents/A4/S2/Web%20Datamining/SCHUTTERLE_Justine_POUPAT_Emilie_PROJET_Nobel/results_distmult/training_triples
INFO:pykeen.pipeline.api:Saved to directory: C:\Users\user\On

DistMult done.


In [30]:
# --- Evaluation Metrics ---
def get_metrics(result, model_name):
    metrics   = result.metric_results.to_dict()
    both      = metrics.get("both", {})
    realistic = both.get("realistic", {})
    return {
        "Model":   model_name,
        "MRR":     round(realistic.get("inverse_harmonic_mean_rank", 0), 4),
        "Hits@1":  round(realistic.get("hits_at_1",  0), 4),
        "Hits@3":  round(realistic.get("hits_at_3",  0), 4),
        "Hits@10": round(realistic.get("hits_at_10", 0), 4),
    }

rows = [
    get_metrics(result_transe,   "TransE"),
    get_metrics(result_distmult, "DistMult"),
]
metrics_df = pd.DataFrame(rows)
print("=== Evaluation Results (Filtered Metrics, Both Head+Tail) ===")
print(metrics_df.to_string(index=False))
print()
print("Note: Low MRR is expected with small KB (<50k triples).")
print("This is discussed in the Critical Reflection section.")
metrics_df.to_csv("kge_evaluation_results.csv", index=False)


=== Evaluation Results (Filtered Metrics, Both Head+Tail) ===
   Model    MRR  Hits@1  Hits@3  Hits@10
  TransE 0.0837  0.0265  0.0884   0.1901
DistMult 0.2028  0.1181  0.2175   0.3783

Note: Low MRR is expected with small KB (<50k triples).
This is discussed in the Critical Reflection section.


In [43]:
# --- 5.2 KB Size Sensitivity ---
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
import numpy as np

size_results = []
AVAILABLE = len(train_triples)

# Use fractions of available data instead of fixed sizes
targets = [int(0.5 * AVAILABLE), int(0.75 * AVAILABLE), AVAILABLE]
labels  = ["50%", "75%", "full"]

for target_size, label in zip(targets, labels):
    actual = min(target_size, AVAILABLE)
    subset = train_triples[:actual]
    print(f"Training TransE on {label} ({actual:,} triples)...")

    # Build factory from subset only — DO NOT share entity_to_id with tf_test
    # to avoid 'index out of range' when subset has fewer entities
    tf_sub = TriplesFactory.from_labeled_triples(
        triples=np.array(subset),
    )
    # Build a compatible test set using only entities known to tf_sub
    known_entities  = set(tf_sub.entity_to_id.keys())
    known_relations = set(tf_sub.relation_to_id.keys())
    test_filtered   = [
        t for t in test_triples
        if t[0] in known_entities and t[2] in known_entities
        and t[1] in known_relations
    ]
    if len(test_filtered) < 10:
        print(f"  Skipped: not enough compatible test triples ({len(test_filtered)})")
        size_results.append({"Subset": label, "Triples": actual,
                             "Error": "insufficient test triples",
                             "Model": f"TransE-{label}",
                             "MRR": float("nan"), "Hits@1": float("nan"),
                             "Hits@3": float("nan"), "Hits@10": float("nan")})
        continue

    tf_test_sub = TriplesFactory.from_labeled_triples(
        triples=np.array(test_filtered),
        entity_to_id=tf_sub.entity_to_id,
        relation_to_id=tf_sub.relation_to_id,
    )

    try:
        r = pipeline(
            training=tf_sub, testing=tf_test_sub,
            model="TransE",
            model_kwargs=dict(embedding_dim=50),
            optimizer_kwargs=dict(lr=0.01),
            training_kwargs=dict(num_epochs=50,
                                 batch_size=min(256, actual // 2)),
            negative_sampler="basic",
            random_seed=42, device="cpu",
        )
        m = get_metrics(r, f"TransE-{label}")
        m["Triples"] = actual
        m["Subset"]  = label
        m["Error"]   = float("nan")
        size_results.append(m)
        print(f"  MRR={m['MRR']:.4f}, Hits@10={m['Hits@10']:.4f}")
    except Exception as e:
        print(f"  Error: {e}")
        size_results.append({"Subset": label, "Triples": actual,
                             "Error": str(e), "Model": f"TransE-{label}",
                             "MRR": float("nan"), "Hits@1": float("nan"),
                             "Hits@3": float("nan"), "Hits@10": float("nan")})

import pandas as pd
size_df = pd.DataFrame(size_results)
print("\n=== Size Sensitivity Results ===")
print(size_df[["Subset", "Triples", "Model", "MRR", "Hits@10"]].to_string(index=False))
size_df.to_csv("size_sensitivity_results.csv", index=False)

INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Training TransE on 50% (10,898 triples)...


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Training epochs on cpu: 100%|██████████| 50/50 [00:19<00:00,  2.51epoch/s, loss=0.0591, prev_loss=0.0587]
Evaluating on cpu:   0%|          | 0.00/2.05k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 2.05k/2.05k [00:03<00:00, 519triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 4.02s seconds
INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


  MRR=0.0898, Hits@10=0.1813
Training TransE on 75% (16,347 triples)...


INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
Training epochs on cpu: 100%|██████████| 50/50 [00:29<00:00,  1.70epoch/s, loss=0.0734, prev_loss=0.0751]
Evaluating on cpu:   0%|          | 0.00/2.34k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 2.34k/2.34k [00:05<00:00, 411triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 5.82s seconds


  MRR=0.0872, Hits@10=0.1861
Training TransE on full (21,796 triples)...


INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
Training epochs on cpu: 100%|██████████| 50/50 [00:37<00:00,  1.32epoch/s, loss=0.0818, prev_loss=0.0869]
Evaluating on cpu:   0%|          | 0.00/2.53k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 2.53k/2.53k [00:07<00:00, 360triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 7.20s seconds


  MRR=0.0883, Hits@10=0.1896

=== Size Sensitivity Results ===
Subset  Triples       Model    MRR  Hits@10
   50%    10898  TransE-50% 0.0898   0.1813
   75%    16347  TransE-75% 0.0872   0.1861
  full    21796 TransE-full 0.0883   0.1896


In [32]:
# --- 6.1 Nearest Neighbors ---
import torch
from torch.nn.functional import normalize

model_transe     = result_transe.model
entity_embs      = model_transe.entity_representations[0](indices=None).detach().cpu()
entity_embs_norm = normalize(entity_embs, dim=1)
e2id             = tf_train.entity_to_id
id2e             = {v: k for k, v in e2id.items()}

# Sample some well-known Nobel entities from our KB
SAMPLE_URIS = [
    "http://www.wikidata.org/entity/Q593",    # Marie Curie
    "http://www.wikidata.org/entity/Q9696",   # Albert Einstein
    "http://www.wikidata.org/entity/Q7191",   # Prix Nobel
]

print("=== 6.1 Nearest Neighbors in Embedding Space ===")
for uri in SAMPLE_URIS:
    if uri not in e2id:
        print(f"  {uri.split('/')[-1]} not in training set — skipping")
        continue
    idx     = e2id[uri]
    vec     = entity_embs_norm[idx].unsqueeze(0)
    sims    = (entity_embs_norm @ vec.T).squeeze()
    top_idx = sims.argsort(descending=True)[1:6].tolist()
    short   = uri.split("/")[-1]
    print(f"\n  Entity: {short}")
    for i in top_idx:
        print(f"    -> {id2e[i].split('/')[-1]} (sim={sims[i]:.3f})")


=== 6.1 Nearest Neighbors in Embedding Space ===

  Entity: Q593
    -> Q602358 (sim=0.543)
    -> Q4532138 (sim=0.541)
    -> Q14916674 (sim=0.539)
    -> Q66386517 (sim=0.527)
    -> Q113510146 (sim=0.501)

  Entity: Q9696
    -> Q49325 (sim=0.649)
    -> Q165421 (sim=0.644)
    -> Q6780521 (sim=0.636)
    -> Q134549 (sim=0.628)
    -> Q272401 (sim=0.618)

  Entity: Q7191
    -> Q32360670 (sim=0.545)
    -> Q950604 (sim=0.544)
    -> Q5043 (sim=0.534)
    -> Q6616 (sim=0.494)
    -> Q13206142 (sim=0.491)


In [33]:
# --- 6.2 t-SNE Clustering ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from matplotlib.patches import Patch

MAX_POINTS = 2000
emb_np   = entity_embs.numpy()
all_uris = [id2e[i] for i in range(len(id2e))]

if len(all_uris) > MAX_POINTS:
    idx_sample = random.sample(range(len(all_uris)), MAX_POINTS)
    emb_sample = emb_np[idx_sample]
    uri_sample = [all_uris[i] for i in idx_sample]
else:
    emb_sample = emb_np
    uri_sample = all_uris

tsne    = TSNE(n_components=2, random_state=42, perplexity=min(30, len(uri_sample)//4))
coords  = tsne.fit_transform(emb_sample)

# Color by URI prefix (Nobel private vs Wikidata)
def get_color(uri):
    if "nobel-kb.org" in uri:
        return "red"     # Private Nobel KB entities
    elif "wikidata.org/entity/Q" in uri:
        return "steelblue"  # Wikidata entities
    else:
        return "gray"

colors = [get_color(u) for u in uri_sample]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, alpha=0.5, s=8)
ax.set_title("t-SNE of Entity Embeddings (TransE) — Prix Nobel KB")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
legend = [
    Patch(color="red",      label="Private Nobel entities"),
    Patch(color="steelblue", label="Wikidata entities"),
    Patch(color="gray",      label="Other URIs"),
]
ax.legend(handles=legend)
plt.tight_layout()
plt.savefig("tsne_embeddings.png", dpi=120)
plt.close()
print("t-SNE plot saved to tsne_embeddings.png")


t-SNE plot saved to tsne_embeddings.png


In [34]:
# --- 6.3 Relation Pattern Analysis ---
from collections import Counter, defaultdict

triple_set   = set(triples_filtered)
sym_counts   = defaultdict(int)
total_counts = defaultdict(int)

for s, p, o in triples_filtered:
    total_counts[p] += 1
    if (o, p, s) in triple_set:
        sym_counts[p] += 1

print(f"{'Relation':55s} {'Total':>6} {'Sym%':>6} {'Pattern'}")
print("-" * 80)
for p in sorted(total_counts, key=lambda x: -total_counts[x])[:15]:
    short   = p.split("/")[-1]
    total   = total_counts[p]
    sym_pct = 100 * sym_counts[p] / total if total > 0 else 0
    if sym_pct > 60:
        pattern = "SYMMETRIC"
    elif sym_pct < 5:
        pattern = "ASYMMETRIC"
    else:
        pattern = "MIXED"
    print(f"  {short[:53]:55s} {total:>6} {sym_pct:>5.0f}% {pattern}")

print()
print("Discussion:")
print("  SYMMETRIC  -> DistMult handles well, TransE struggles")
print("  ASYMMETRIC -> TransE handles well (translational model)")


Relation                                                 Total   Sym% Pattern
--------------------------------------------------------------------------------
  P31                                                       1803     0% ASYMMETRIC
  P1343                                                     1685     0% ASYMMETRIC
  P106                                                      1533     0% ASYMMETRIC
  extractedFrom                                             1468     0% ASYMMETRIC
  22-rdf-syntax-ns#type                                     1459     0% ASYMMETRIC
  P166                                                      1350     0% ASYMMETRIC
  P463                                                      1292     0% ASYMMETRIC
  P530                                                      1050    14% MIXED
  P17                                                        866     3% ASYMMETRIC
  P1346                                                      536     0% ASYMMETRIC
  P460          

In [35]:
# --- Exercise 8: SWRL rule vs KGE vectors ---
relation_embs = result_transe.model.relation_representations[0](indices=None).detach().cpu()
r2id          = tf_train.relation_to_id
rel_norm      = normalize(relation_embs, dim=1)

print("=== Exercise 8: Rule-based vs Embedding-based ===")
print()
print("SWRL Rule (Prix Nobel domain):")
print("  NobelLaureate(?p) ∧ prop:receives(?p, ?prize) ∧ NobelPrize(?prize)")
print("  → NobelAwardee(?p)")
print()
print("Equivalent KGE vector question:")
print("  Is vector(receives) + vector(NobelPrize) ≈ vector(NobelAwardee)?")
print()

# Find relevant relation vectors
PROP_RECEIVE = "http://nobel-kb.org/property/receives"
WDT_P166     = "http://www.wikidata.org/prop/direct/P166"  # award received

for target_rel, name in [(PROP_RECEIVE, "prop:receives"), (WDT_P166, "wdt:P166")]:
    if target_rel in r2id:
        idx = r2id[target_rel]
        vec = rel_norm[idx]
        sims = (rel_norm @ vec.unsqueeze(1)).squeeze()
        top = sims.argsort(descending=True)[1:4].tolist()
        id2r = {v: k for k, v in r2id.items()}
        print(f"  Nearest relations to {name}:")
        for i in top:
            print(f"    -> {id2r[i].split('/')[-1]} (sim={sims[i]:.3f})")
    else:
        print(f"  {name} not in training set (small KB — this relation may not have appeared enough)")


=== Exercise 8: Rule-based vs Embedding-based ===

SWRL Rule (Prix Nobel domain):
  NobelLaureate(?p) ∧ prop:receives(?p, ?prize) ∧ NobelPrize(?prize)
  → NobelAwardee(?p)

Equivalent KGE vector question:
  Is vector(receives) + vector(NobelPrize) ≈ vector(NobelAwardee)?

  Nearest relations to prop:receives:
    -> P1891 (sim=0.275)
    -> P3602 (sim=0.228)
    -> P1877 (sim=0.212)
  Nearest relations to wdt:P166:
    -> P1411 (sim=0.602)
    -> P611 (sim=0.458)
    -> P1576 (sim=0.449)


---
## Part 3 — SWRL Rule on the Nobel KB

Règle Horn SWRL à 2 conditions sur notre ontologie Nobel :
```
Person(?p) ∧ prop:receives(?p, ?prize) ∧ NobelPrize(?prize) → NobelLaureate(?p)
```
Toute personne qui reçoit un Prix Nobel est inférée comme `NobelLaureate`.

In [36]:
from owlready2 import *
import os

# Build Nobel ontology in memory
nobel_onto = get_ontology("http://nobel-kb.org/ontology#")

with nobel_onto:
    class NobelPrize(Thing):
        """A Nobel Prize award."""
        pass

    class Person(Thing):
        """A person (potential laureate)."""
        pass

    class NobelLaureate(Person):
        """A person who has received a Nobel Prize."""
        pass

    class receives(ObjectProperty):
        domain = [Person]
        range  = [NobelPrize]

    # Sample individuals
    PhysicsPrize = NobelPrize("NobelPrizePhysics")
    ChemPrize    = NobelPrize("NobelPrizeChemistry")
    PeacePrize   = NobelPrize("NobelPrizePeace")

    Marie       = Person("MarieCurie")
    Einstein    = Person("AlbertEinstein")
    Malala      = Person("MalalaYousafzai")
    RandomGuy   = Person("JohnDoePerson")  # Does NOT receive a Nobel prize

    Marie.receives    = [PhysicsPrize, ChemPrize]
    Einstein.receives = [PhysicsPrize]
    Malala.receives   = [PeacePrize]
    # RandomGuy receives nothing -> should NOT be inferred as NobelLaureate

    # SWRL rule: Person(?p) ∧ receives(?p, ?prize) ∧ NobelPrize(?prize) → NobelLaureate(?p)
    rule = Imp()
    rule.set_as_rule("""
        Person(?p), receives(?p, ?prize), NobelPrize(?prize) -> NobelLaureate(?p)
    """)

print("Nobel ontology created with SWRL rule.")
print("Individuals:", [ind.name for ind in Person.instances()])
print("SWRL rule body:", rule.body)


Nobel ontology created with SWRL rule.
Individuals: ['MarieCurie', 'AlbertEinstein', 'MalalaYousafzai', 'JohnDoePerson']
SWRL rule body: [Person(?p), receives(?p, ?prize), NobelPrize(?prize)]


In [40]:
# --- Run reasoner on Nobel KB ---
from pathlib import Path

temp_dir = Path("C:/tmp_owl")
temp_dir.mkdir(exist_ok=True)
owl_out = str(temp_dir / "nobel_onto.owl")
nobel_onto.save(file=owl_out, format="rdfxml")

nobel_onto2 = get_ontology("file:///C:/tmp_owl/nobel_onto.owl")
nobel_onto2.load(fileobj=open(owl_out, "rb"))

print("Running Pellet reasoner on Nobel KB...")
with nobel_onto2:
    sync_reasoner_pellet(infer_property_values=True)
print("Reasoning complete.")

Running Pellet reasoner on Nobel KB...


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-pac

* Owlready * Adding relation family.Pedro isChildOf family.John
* Owlready * Adding relation family.Paul isChildOf family.Peter
* Owlready * Adding relation family.Michael isChildOf family.Alex
* Owlready * Adding relation family.Michael isChildOf family.Thomas
* Owlready * Adding relation family.Tom isChildOf family.Alex
* Owlready * Adding relation family.Tom isChildOf family.Thomas
* Owlready * Adding relation family.Chloé isChildOf family.Peter
* Owlready * Adding relation family.Chloé isChildOf family.Marie
* Owlready * Adding relation family.Sylvie isParentOf family.Claude
* Owlready * Adding relation family.Sylvie isChildOf family.Peter
* Owlready * Adding relation family.Sylvie isChildOf family.Marie
* Owlready * Adding relation family.Alex isParentOf family.Michael
* Owlready * Adding relation family.Alex isParentOf family.Tom
* Owlready * Adding relation family.John isParentOf family.Pedro
* Owlready * Adding relation family.Thomas isParentOf family.Michael
* Owlready * Addin

* Owlready2 * Pellet took 1.2051975727081299 seconds
* Owlready * Reparenting ontology.MalalaYousafzai: {ontology.Person} => {ontology.NobelLaureate}
* Owlready * Reparenting ontology.AlbertEinstein: {ontology.Person} => {ontology.NobelLaureate}
* Owlready * Reparenting ontology.MarieCurie: {ontology.Person} => {ontology.NobelLaureate}
* Owlready * Reparenting family.Daughter: {family.Child} => {family.Female, family.Child}
* Owlready * Reparenting family.Chloé: {family.Female} => {family.OldPerson, family.Daughter}
* Owlready * Reparenting family.Sylvie: {family.Female} => {family.OldPerson, family.Daughter}
* Owlready * Reparenting family.Claude: {family.Female} => {family.OldPerson, family.Daughter}
* Owlready * Reparenting family.Son: {family.Child} => {family.Male, family.Child}
* Owlready * Reparenting family.Pedro: {family.Male} => {family.Son, family.OldPerson}
* Owlready * Reparenting family.Thomas: {family.Male} => {family.Son, family.OldPerson, family.Father}
* Owlready * Re

In [41]:
# --- Results ---
print("=== SWRL Results on Nobel KB ===")
print("Inferred as NobelLaureate:")
found_laureates = []
for cls in nobel_onto2.classes():
    if cls.name == "NobelLaureate":
        for ind in cls.instances():
            print(f"  -> {ind.name} ✓")
            found_laureates.append(ind.name)

print()
print("Expected    : MarieCurie, AlbertEinstein, MalalaYousafzai")
print("NOT expected: JohnDoePerson (no receives relation)")
print()

# Verify
expected = {"MarieCurie", "AlbertEinstein", "MalalaYousafzai"}
got      = set(found_laureates)
print(f"Correct inferences: {expected & got}")
print(f"Missing           : {expected - got}")
print(f"Unexpected        : {got - expected}")


=== SWRL Results on Nobel KB ===
Inferred as NobelLaureate:
  -> MarieCurie ✓
  -> AlbertEinstein ✓
  -> MalalaYousafzai ✓

Expected    : MarieCurie, AlbertEinstein, MalalaYousafzai
NOT expected: JohnDoePerson (no receives relation)

Correct inferences: {'MarieCurie', 'MalalaYousafzai', 'AlbertEinstein'}
Missing           : set()
Unexpected        : set()


**Summary of Part 3 — SWRL on Nobel KB:**

La règle SWRL `Person(?p) ∧ receives(?p, ?prize) ∧ NobelPrize(?prize) → NobelLaureate(?p)` infère correctement les lauréats :
- Marie Curie → NobelLaureate (reçoit Prix Nobel de physique ET de chimie)
- Albert Einstein → NobelLaureate
- Malala Yousafzai → NobelLaureate
- John Doe → **non** inféré (aucune relation `receives` vers un `NobelPrize`)

**Lien avec Exercise 8 (KGE) :** L'équivalent en embedding serait de vérifier que `vector(receives) + vector(NobelPrize) ≈ vector(NobelLaureate)`. Dans TransE, cela correspond à tester si la translation dans l'espace vectoriel encode cette relation ternaire.
